# Data Processing

This notebook processes the raw CSV files into a more usable format to be used in further data analysis.

The raw data is stored as 2 columns, FEN and evaluation. These are not practical to work with and need to be processed. This notebook parses the FEN and extracts relevant information out of it to be used later. The evaluations are normalized to integers, explained below.

## FEN format
A FEN is a short computer readable representation of the chessboard. A sample modern FEN could be:
`1r6/5pp1/R1R4p/1r1pP3/2pkQPP1/7P/1P6/2K5 w - - 0 41`

After the starting position and moves e4, c5, Nf3, the FEN is:

`rnbqkbnr/pp1ppppp/8/2p5/4P3/5N2/PPPP1PPP/RNBQKB1R b KQkq - 1 2`

The FEN is whitespace separated, and contains the following information in order from left to right:
- Board position (explained below)
- Player turn (w = white, b = black)
- Castling availability (explained below)
- En passant target square
- Halfmove clock (number of moves since last pawn move or capture, used for stalemate by 50 move rule)
- Fullmove number (number of full moves since the beginning of the game)

A `-` means the data is not applicable in this position.

#### Board Position
Board position is represented as a single string, read from top left of the board to the bottom right. Each character represents a certain piece or absence of piece. Uppercase characters are for white, lowercase for black
- P/p - pawn
- R/r - rook
- N/n - knight
- B/b - bishop
- Q/q - queen
- K/k - king
- / - next rank (row)
- 1-8 - number of empty squares

#### Castling availability
The presence of any of these characters means the right to castle is kept. If the character is absent, the player has already lost the right to castle in that direction:

- K - white kingside
- Q - white queenside
- k - black kingside
- q - black queenside

## Evaluation normalization
The evaluation is normally represented as a integer, positive meaning advantage for white, and negative meaning advantage for black. However, an evaluation beginning with `M` means that there is a forced mate in that many moves (eg `M+5` or `M-5` means forced mate in 5 for white and black respectively). Evaluation in this format is unusable for data analytics, so mate values have been normalized to a very high integer outside the range of all other evaluations.

## Data processing output
With all this in mind, the original CSVs have been processed to new CSVs that contain the following columns:
- P,N,B,R,Q,p,n,b,r,q - a column each for piece count of the respective pieces
- white_to_move - 1 for white, 0 for black
- white_kingside - can white castle kingside
- white_queenside - can white castle queenside
- black_kingside - can black castle kingside
- black_queenside - can black castle queenside
- halfMoves - number of half moves
- fullMoves - number of full moves
- Evaluation - normalized evaluation
- white_material - added up value of all white's pieces
- black_material - added up value of all black's pieces
- material_diff - material difference (positive meaning white has more material than black, negative vice versa)

## To run
Please run all the blocks in this notebook in order to prevent errors. The processing might take a while due to large CSV sizes involved.

In [6]:
import pandas as pd
from collections import Counter

print("Imports successful")

Imports successful


In [7]:
# normalization value, way above all other evaluations, but not too high to distort graphs
NORMALIZED_MATE_EVALUATION = 16000
def process_eval(evaluation: str):
    try:
        return int(evaluation) # if already an int, return
    except ValueError: # ValueError is thrown if unable to convert to int, meaning the evaluation must be mate
        num = int(evaluation.removeprefix('#')) # remove the mate prefix and return the normalized value
        if num > 0:
            return NORMALIZED_MATE_EVALUATION-num
        else:
            return -NORMALIZED_MATE_EVALUATION-num

# normalize the evaluations
def normalize_evaluation(evaluations : pd.Series):
    return [process_eval(evaluation) for evaluation in evaluations]

In [8]:
PIECES = 'PNBRQpnbrq'  # uppercase = white, lowercase = black, king omitted
VALUES = {'P': 1, 'N': 3, 'B': 3, 'R': 5, 'Q': 9} # piece values
CHUNK_SIZE = 500_000 # number of rows processed at a time, reduce RAM usage
def process_csv(file: str):
    print("Importing: " + file)

    output_file = file[:-4] + '_formatted.csv' # output file name
    header_chunk = True # first chunk includes headers, the rest don't
    chunk_num = 1 # num of chunk for printing

    for csv in pd.read_csv(file, chunksize=CHUNK_SIZE): # read the CSV file in chunks
        rows = [] # create list
        for fen in csv['FEN']: # for each row fen in CSV file
            placement, player, castling, _ep, half, full = fen.split(' ') # split fen into substrings separated by a ' ' and give them a name
            counts = Counter(placement) # counts occurrences of every character inside the string
            rows.append( # add parsed row
                [counts.get(p, 0) for p in PIECES] # add counts of pieces
                + [int(player == 'w')] # add player as int (White = 1, Black = 0)
                + [int(c in castling) for c in 'KQkq'] # add all 4 castling rules as ints (1 = true, 0 = false)
                + [int(half), int(full)] # add number of half moves for stalemate and full moves played
            )

        # convert standard list to dataframe
        new_data = pd.DataFrame(
            rows, # the data from above
            columns=list(PIECES) + [ # piece column labels by piece, then other labels corresponding to the data above
                'white_to_move',
                'white_kingside', 'white_queenside', 'black_kingside', 'black_queenside',
                'halfMoves', 'fullMoves',
            ],
        )
        new_data['Evaluation'] = normalize_evaluation(csv['Evaluation']) # normalize the evaluation and add to dataframe

        # add up material value and add to dataframe
        new_data['white_material'] = sum(new_data[p] * v for p, v in VALUES.items())
        new_data['black_material'] = sum(new_data[p.lower()] * v for p, v in VALUES.items())
        new_data['material_diff'] = new_data['white_material'] - new_data['black_material'] # add material difference to data frame

        new_data.to_csv(
            output_file, # save to file
            index=False, # no index
            mode='w' if header_chunk else 'a', # write mode for first chunk, append for all other chunks
            header=header_chunk # add a header if at the first chunk
        )

        header_chunk = False
        print("Processed "+str(CHUNK_SIZE*chunk_num)+" rows")
        chunk_num += 1
    print(file + " imported to " + output_file)

In [9]:
print("This might take a while, depending on single threaded CPU speed")
print("The CSVs are processed in chunks of "+str(CHUNK_SIZE)+" to reduce RAM usage")
process_csv('chessData.csv')
process_csv('random_evals.csv')
process_csv('tactic_evals.csv')

This might take a while, depending on single threaded CPU speed
The CSVs are processed in chunks of 500000 to reduce RAM usage
Importing: chessData.csv
Processed 500000 rows
Processed 1000000 rows
Processed 1500000 rows
Processed 2000000 rows
Processed 2500000 rows
Processed 3000000 rows
Processed 3500000 rows
Processed 4000000 rows
Processed 4500000 rows
Processed 5000000 rows
Processed 5500000 rows
Processed 6000000 rows
Processed 6500000 rows
Processed 7000000 rows
Processed 7500000 rows
Processed 8000000 rows
Processed 8500000 rows
Processed 9000000 rows
Processed 9500000 rows
Processed 10000000 rows
Processed 10500000 rows
Processed 11000000 rows
Processed 11500000 rows
Processed 12000000 rows
Processed 12500000 rows
Processed 13000000 rows
chessData.csv imported to chessData_formatted.csv
Importing: random_evals.csv
Processed 500000 rows
Processed 1000000 rows
Processed 1500000 rows
random_evals.csv imported to random_evals_formatted.csv
Importing: tactic_evals.csv
Processed 5000